In [ ]:
"""Notebook-ready opening first-45-minute L1 pressure-return interaction."""

from __future__ import annotations


def main(datasources, start_date, end_date):
    import dai
    import pandas as pd

    table_name = datasources["bar1m"]
    pressure = """
        (
            CAST(bid_volume1 AS DOUBLE) - CAST(ask_volume1 AS DOUBLE)
        )
        / NULLIF(
            CAST(bid_volume1 AS DOUBLE) + CAST(ask_volume1 AS DOUBLE),
            0
        )
    """
    minute_return = """
        (
            CAST(close AS DOUBLE) - CAST(open AS DOUBLE)
        )
        / NULLIF(CAST(open AS DOUBLE), 0)
    """
    sql = f"""
    select
        date::DATE::DATETIME AS date,
        instrument,
        AVG(
            CASE
                WHEN date::TIME <= TIME '10:15:00'
                THEN ({pressure}) * ({minute_return})
                ELSE NULL
            END
        ) AS factor
    from {table_name}
    GROUP BY date::DATE, instrument
    ORDER BY date, instrument
    """
    factor = dai.query(
        sql,
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()
    factor["date"] = pd.to_datetime(factor["date"]).dt.normalize()
    factor["factor"] = pd.to_numeric(factor["factor"], errors="coerce")

    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df()
    stk_pool["date"] = pd.to_datetime(stk_pool["date"]).dt.normalize()

    return (
        pd.merge(factor, stk_pool, how="inner", on=["date", "instrument"])
        .dropna(subset=["factor"])
        .sort_values(["date", "instrument"])
        .reset_index(drop=True)
        .loc[:, ["date", "instrument", "factor"]]
    )
